In [10]:
import os
import re
import pandas as pd

MODEL_PREFIX = "meta-llama_llama-3.3-70b-instruct"

added_vars_map = {
    "baseline": 0,
    "baseline_plus_5": 5,
    "baseline_plus_10": 10,
    "baseline_plus_15": 15,
    "baseline_plus_20": 20,
    "baseline_plus_25": 25,
    "baseline_plus_30": 30,
    "baseline_plus_35": 35,
    "baseline_plus_40": 40,
    "baseline_plus_45": 45,
    "baseline_plus_50": 50,
}

prompt_types = ["neutral", "logical", "human_impact", "clinical_judgement"]

num_patients = 50
sim_reps = 5

for dataset_name in added_vars_map:
    for prompt_name in prompt_types:
        folder = f"{MODEL_PREFIX}_{dataset_name}_{prompt_name}"
        if not os.path.isdir(folder):
            print(f"Skipping missing folder: {folder}")
            continue

        results = []

        for patient_idx in range(num_patients):
            for sim_idx in range(sim_reps):
                fname = f"output_patient{patient_idx}_sim{sim_idx}.txt"
                fpath = os.path.join(folder, fname)

                if not os.path.exists(fpath):
                    print(f"  Missing: {fpath}")
                    continue

                with open(fpath, "r", encoding="utf-8") as f:
                    result = f.read()

                out = {
                    "Dataset": dataset_name,
                    "Prompt": prompt_name,
                    "Patient_ID": patient_idx,
                    "Simulation_Number": sim_idx,
                }

                text = result.replace("–", "-").strip()
                lines = [line.strip() for line in text.splitlines() if line.strip()]

                # 1. Risk score — handles three formats:
                #    a) "Risk Assessment Score: 7"  (same line)
                #    b) "7" alone on the first line (logical prompt)
                #    c) Score on the line after the label
                score_val = None

                # format b) — bare number or "1. 4.5" on first line
                try:
                    first = re.sub(r'^\d+\.\s*', '', lines[0])  # strip leading "1. "
                    first = first.replace("**", "")
                    score_val = float(first)
                except (ValueError, IndexError):
                    pass

                # fix: treat 0 as valid score (float(0) is falsy)
                # so use "is None" checks throughout, not truthiness

                # formats a) and c) — labelled (strip bold markdown before matching)
                if score_val is None:
                    for i, line in enumerate(lines):
                        clean_line = line.replace("**", "")
                        if any(kw in clean_line for kw in ["Risk Assessment Score", "Final Hospitalization Risk Score", "Risk Score", "risk score"]):
                            match = re.search(r':\s*(\d+\.?\d*)', clean_line)
                            if match:
                                score_val = float(match.group(1))
                                break
                            for j in range(i + 1, min(i + 4, len(lines))):
                                try:
                                    score_val = float(lines[j].replace("**", ""))
                                    break
                                except ValueError:
                                    continue
                            break
                out["Risk_Assessment_Score"] = score_val

                # 2. Parameter table
                start = None
                for i, line in enumerate(lines):
                    if "Parameter" in line and "Value" in line:
                        start = i + 1
                        break
                if start is not None:
                    for line in lines[start:]:
                        if not line.startswith("|"):
                            break
                        parts = [p.strip() for p in line.split("|") if p.strip()]
                        if len(parts) == 2:
                            name, val = parts
                            try:
                                out[name] = float(val)
                            except ValueError:
                                pass

                # 3. Rationale
                rationale = []
                capture = False
                for line in lines:
                    if "Rationale" in line or "Justification" in line:
                        capture = True
                        continue
                    if capture:
                        rationale.append(line)
                out["Rationale"] = " ".join(rationale)

                results.append(out)

        if results:
            df_out = pd.DataFrame(results)
            df_out.to_csv(os.path.join(folder, "checkpoint.csv"), index=False)
            null_count = df_out["Risk_Assessment_Score"].isna().sum()
            print(f"Rebuilt {folder} — {len(df_out)} rows, {null_count} missing scores")
        else:
            print(f"No results found in {folder}")

Rebuilt meta-llama_llama-3.3-70b-instruct_baseline_neutral — 250 rows, 0 missing scores
Rebuilt meta-llama_llama-3.3-70b-instruct_baseline_logical — 250 rows, 0 missing scores
Rebuilt meta-llama_llama-3.3-70b-instruct_baseline_human_impact — 250 rows, 0 missing scores
Rebuilt meta-llama_llama-3.3-70b-instruct_baseline_clinical_judgement — 250 rows, 0 missing scores
Rebuilt meta-llama_llama-3.3-70b-instruct_baseline_plus_5_neutral — 250 rows, 0 missing scores
Rebuilt meta-llama_llama-3.3-70b-instruct_baseline_plus_5_logical — 250 rows, 0 missing scores
Rebuilt meta-llama_llama-3.3-70b-instruct_baseline_plus_5_human_impact — 250 rows, 0 missing scores
Rebuilt meta-llama_llama-3.3-70b-instruct_baseline_plus_5_clinical_judgement — 250 rows, 0 missing scores
Rebuilt meta-llama_llama-3.3-70b-instruct_baseline_plus_10_neutral — 250 rows, 0 missing scores
Rebuilt meta-llama_llama-3.3-70b-instruct_baseline_plus_10_logical — 250 rows, 0 missing scores
Rebuilt meta-llama_llama-3.3-70b-instruct_ba

In [11]:
import pandas as pd
from collections import defaultdict

MODEL_PREFIX = "meta-llama_llama-3.3-70b-instruct"

added_vars_map = {
    "baseline": 0,
    "baseline_plus_5": 5,
    "baseline_plus_10": 10,
    "baseline_plus_15": 15,
    "baseline_plus_20": 20,
    "baseline_plus_25": 25,
    "baseline_plus_30": 30,
    "baseline_plus_35": 35,
    "baseline_plus_40": 40,
    "baseline_plus_45": 45,
    "baseline_plus_50": 50,
}

prompt_types = ["neutral", "logical", "human_impact", "clinical_judgement"]

data_by_prompt = defaultdict(list)

for dataset_name, added_vars in added_vars_map.items():
    for prompt_name in prompt_types:
        folder = f"{MODEL_PREFIX}_{dataset_name}_{prompt_name}"
        checkpoint = os.path.join(folder, "checkpoint.csv")

        if not os.path.exists(checkpoint):
            print(f"Missing: {checkpoint}")
            continue

        df = pd.read_csv(checkpoint)

        for _, row in df.iterrows():
            data_by_prompt[prompt_name].append({
                "Patient_ID":  f"P{int(row['Patient_ID']):02d}",
                "Condition":   dataset_name,
                "Added_Vars":  added_vars,
                "Trial":       int(row["Simulation_Number"]) + 1,
                "Risk_Score":  row["Risk_Assessment_Score"]
            })

# save one CSV per prompt type
for prompt_name, records in data_by_prompt.items():
    out_df = pd.DataFrame(records).sort_values(["Patient_ID", "Condition", "Trial"])
    fname = f"scores_{MODEL_PREFIX}_{prompt_name}.csv"
    out_df.to_csv(fname, index=False)
    print(f"Saved {fname} — {len(out_df)} rows, {out_df['Risk_Score'].isna().sum()} missing")

Saved scores_meta-llama_llama-3.3-70b-instruct_neutral.csv — 2750 rows, 0 missing
Saved scores_meta-llama_llama-3.3-70b-instruct_logical.csv — 2750 rows, 0 missing
Saved scores_meta-llama_llama-3.3-70b-instruct_human_impact.csv — 2750 rows, 0 missing
Saved scores_meta-llama_llama-3.3-70b-instruct_clinical_judgement.csv — 2750 rows, 0 missing
